In [73]:
import pandas as pd
import numpy as np

In [85]:
df_final = pd.read_excel('./../../data/dados_tratados/df_rel_tratado.xlsx')

df_ipca = pd.read_excel('./../../data/raw/df_ipca.xlsx')

df_amort_extr = pd.read_csv('./../../data/dados_tratados/amort_extraord.csv')

In [89]:
df_final["data_aniversario"] = df_final["data_aniversario"].dt.to_period("M")
df_ipca["data"] = df_ipca["data"].dt.to_period("M")

AttributeError: 'PeriodProperties' object has no attribute 'to_period'

In [91]:
df_final["mes_razao_usada"] = df_final["data_aniversario"] - 2

In [93]:
df_final = df_final.merge(
    df_ipca[["data", "razao_ni"]],
    left_on="mes_razao_usada",
    right_on="data",
    how="left",
    suffixes=("", "razao_ni")
)

df_final = df_final.drop(columns=['data', 'proxima_data'])

In [95]:
df_final["razao_ni"] = df_final["razao_ni"].clip(lower=1)

df_final["fator_ipca"] = np.where(
    df_final["razao_ni"].notna(),
    df_final["razao_ni"] ** (df_final["dcp"] / df_final["dct"]),
    np.nan
)

In [97]:
df_final['fator_juros'] = (1 + 9 / 100) ** (df_final["dcp"] / 365)

In [99]:
df_final['produtorio_juros'] = df_final["fator_juros"].cumprod()

In [102]:
df_amort_extr['amortizacao_extraordinaria'] = df_amort_extr['amortizacao_extraordinaria'].str.replace(',', '.').astype('float')

In [105]:
df_final['amortizacao_extraordinaria'] = df_amort_extr['amortizacao_extraordinaria']

In [108]:
df_final['fator_ipca'] = df_final['fator_ipca'].fillna(1)

In [111]:
# seu df já pronto
df = df_final.copy()

# valor inicial que você quer colocar na primeira linha
x_inicial = 1002.87

# cria as colunas que serão calculadas
df["vna_inicio"] = 0.0
df["vna_antes_pagamento"] = 0.0
df["juros"] = 0.0
df["vna_depois_pagamento"] = 0.0
df["amortizacao_reais"] = 0.0
df["total_amortizado"] = 0.0

for i in range(1, len(df)):

    if i == 1:
        df.loc[i, "vna_inicio"] = x_inicial
        df.loc[i, "total_amortizado"] = 0
    else:
        df.loc[i, "vna_inicio"] = df.loc[i - 1, "vna_depois_pagamento"]
        df.loc[i, "total_amortizado"] = df.loc[i - 1, "total_amortizado"]

    df.loc[i, "vna_corrigido"] = (
        df.loc[i, "vna_inicio"]
        * df.loc[i, "fator_ipca"]
    )

    df.loc[i, "juros"] = (
        df.loc[i, "vna_corrigido"]
        * (df.loc[i, "fator_juros"] - 1)
    )

    df.loc[i, "vna_antes_pagamento"] = (
        df.loc[i, "vna_corrigido"]
        + df.loc[i, "juros"]
    )

    # percentual amortizado sobre o VNA antes do pagamento
    df.loc[i, "amortizacao_reais"] = (
        df.loc[i, "vna_corrigido"]
        * df.loc[i, "percentual_amortizado"]
    )

    # total amortizado acumulado até essa linha
    df.loc[i, "total_amortizado"] = (
        df.loc[i, "total_amortizado"]
        + df.loc[i, "amortizacao_reais"]
    )

    df.loc[i, "vna_depois_pagamento"] = (
        df.loc[i, "vna_corrigido"]
        - df.loc[i, "amortizacao_reais"]
        - df.loc[i, "amortizacao_extraordinaria"]
    )

In [114]:
df

,Unnamed: 0,data_aniversario,data_pagamento,juros,incorporacao_juros,amortizacao,percentual_amortizado,dcp,dct,mes_razao_usada,...,fator_ipca,fator_juros,produtorio_juros,amortizacao_extraordinaria,vna_inicio,vna_antes_pagamento,vna_depois_pagamento,amortizacao_reais,total_amortizado,vna_corrigido
0,0,2024-06,2024-06-18,0.000000,Sim,Não,0.000000,17.0,31.0,2024-04,...,1.002082,1.004022,1.004022,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
1,1,2024-07,2024-07-16,7.161437,Não,Sim,0.002193,30.0,30.0,2024-05,...,1.004600,1.007108,1.011159,2.50,1002.870000,1014.644915,1002.774066,2.209411,2.209411,1007.483478
2,2,2024-08,2024-08-16,7.142934,Não,Sim,0.002149,30.0,30.0,2024-06,...,1.002100,1.007108,1.018346,5.52,1002.774066,1012.023316,997.200894,2.159488,4.368899,1004.880382
3,3,2024-09,2024-09-17,6.877295,Não,Sim,0.002169,29.0,29.0,2024-07,...,1.003800,1.006870,1.025343,3.57,997.200894,1007.867877,995.249434,2.171149,6.540048,1000.990582
4,4,2024-10,2024-10-16,7.074475,Não,Sim,0.002255,30.0,30.0,2024-08,...,1.000000,1.007108,1.032631,3.57,995.249434,1002.323908,989.435146,2.244287,8.784335,995.249434
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,200,2041-02,2041-02-18,0.316299,Não,Sim,0.197083,27.0,27.0,2040-12,...,1.000000,1.006395,4.030755,0.00,49.459213,49.775512,39.711643,9.747570,1035.598710,49.459213
201,201,2041-03,2041-03-18,0.282280,Não,Sim,0.247526,30.0,30.0,2041-01,...,1.000000,1.007108,4.059407,0.00,39.711643,39.993923,29.881979,9.829664,1045.428375,39.711643
202,202,2041-04,2041-04-16,0.205304,Não,Sim,0.330897,29.0,29.0,2041-02,...,1.000000,1.006870,4.087297,0.00,29.881979,30.087282,19.994122,9.887857,1055.316232,29.881979
203,203,2041-05,2041-05-16,0.142123,Não,Sim,0.498229,30.0,30.0,2041-03,...,1.000000,1.007108,4.116350,0.00,19.994122,20.136245,10.032470,9.961651,1065.277883,19.994122


In [118]:
df_fluxo = pd.DataFrame()

df_fluxo['data'] = df['data_aniversario']
df_fluxo['juros'] = df['juros']
df_fluxo['amortizacao'] = df['amortizacao_extraordinaria'] + df['amortizacao_reais']

In [121]:
df_fluxo["fluxo"] = (
    df_fluxo["juros"].fillna(0)
    + df_fluxo["amortizacao"].fillna(0)
)

df_fluxo["data"] = df_fluxo["data"].dt.to_timestamp() + pd.DateOffset(days=14)

In [124]:
df_fluxo.to_excel('./../../data/dados_tratados/df_fluxo.xlsx')